# 06 — Imagine Text-to-Image Test

Diagnostic notebook for `FluxPipeline` (text-to-image, no ControlNet, no Kontext, no Gradio).

**Pipeline:** `FluxPipeline` — `black-forest-labs/FLUX.1-dev`

**Three explicit tests:**
1. **Diversity** — same prompt, seeds 42 / 12345 / 987654 → pixel hashes must all differ
2. **Repeatability** — same prompt + seed 42, two consecutive runs → pixel hashes must match
3. **Prompt sensitivity** — clearly different prompt, seed 42 → pixel hash must differ from test 1

**Two hashes per image:**
- `pixel_hash` — SHA-256 of raw pixel bytes before PNG encoding (what the model produced in RAM)
- `file_hash` — SHA-256 of the saved .png file (what landed on disk)

This distinguishes: repeated model output (pixel_hash matches) vs filename collision
(different pixel_hash but same file_hash) vs correct behaviour (both differ).

**Filenames:** `06_<label>_seed<N>_<uuid8>.png`

Do not edit `05_Cartoonify_Gradio_Unified.ipynb` until these tests pass.

---

## 1 — Confirm A100 GPU

In [ ]:
!nvidia-smi


## 2 — Install Dependencies

In [ ]:
!pip install --quiet diffusers transformers accelerate sentencepiece safetensors
!pip install --quiet huggingface_hub


In [ ]:
import os
os.kill(os.getpid(), 9)
print('IGNORE: session crashed — continue to the next cell.')


## 3 — Imports

In [ ]:
import gc
import hashlib
import os
import uuid
import torch
from PIL import Image
from diffusers import FluxPipeline
from safetensors import safe_open

print(f'torch : {torch.__version__}')
print(f'CUDA  : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')


## 4 — Mount Google Drive

In [ ]:
import shutil
from google.colab import drive

if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    os.system('fusermount -u /content/drive 2>/dev/null || true')
    shutil.rmtree('/content/drive', ignore_errors=True)

drive.mount('/content/drive')


## 5 — API Tokens

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in to Hugging Face')


## 6 — Configuration

In [ ]:
BASE_MODEL = 'black-forest-labs/FLUX.1-dev'

LORA_SATIREFIC_PATH         = '/content/drive/MyDrive/satirefic/satirefic/satirefic.safetensors'
LORA_SATIREFIC_ADAPTER_NAME = 'satirefic'

OUTPUT_DIR = '/content/drive/MyDrive/cartoonify/outputs/06_imagine_test'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Exact values from Notebook 05 cell-config
GLOBAL_PROMPT_PREFIX = 'crude black and white newspaper caricature'

SATIREFIC_STYLE_BOOST = (
    'maximum grotesque political satire, extreme caricature exaggeration, '
    'absurd anti-authoritarian newspaper cartoon, rough dirty ink linework, '
    'ugly distorted faces, pig-politician symbolism, visual irony, '
    'anarchist press style, flat white paper, brutal black ink, no readable text'
)

print(f'Base model : {BASE_MODEL}')
print(f'Satirefic  : {LORA_SATIREFIC_PATH}')
print(f'Output dir : {OUTPUT_DIR}')


## Setup — Load Model

Loads `FluxPipeline` only — no ControlNet, no Kontext.

⏱️ First download: ~19 GB. Warm cache: ~1 minute.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
_free, _total = torch.cuda.mem_get_info()
print(f'GPU before load: {_free/1e9:.1f} GB free / {_total/1e9:.1f} GB total')
print()

print(f'Loading FluxPipeline from {BASE_MODEL} ...')
pipe = FluxPipeline.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16).to('cuda')

_free2, _ = torch.cuda.mem_get_info()
print(f'GPU after load : {_free2/1e9:.1f} GB free (used {(_free - _free2)/1e9:.1f} GB)')
print(f'Pipeline class : {type(pipe).__name__}')
print('FluxPipeline ready')


## Load LoRA — Inspect and Load Satirefic

Inspects key names from the `.safetensors` file to assess FLUX.1-dev compatibility.
If the file is absent this section prints a skip message and continues.

In [ ]:
lora_exists = os.path.isfile(LORA_SATIREFIC_PATH)
print(f'Path   : {LORA_SATIREFIC_PATH}')
print(f'Exists : {lora_exists}')
print()

if lora_exists:
    with safe_open(LORA_SATIREFIC_PATH, framework='pt', device='cpu') as f:
        metadata = f.metadata()
        all_keys = list(f.keys())

    print('--- Metadata ---')
    if metadata:
        for k, v in metadata.items():
            print(f'  {k}: {v}')
    else:
        print('  (no metadata stored in file)')

    print(f'\n--- Keys: first 30 of {len(all_keys)} total ---')
    for k in all_keys[:30]:
        print(f'  {k}')

    flux_keys = [
        k for k in all_keys
        if any(p in k for p in ['transformer', 'single_transformer', 'lora_transformer'])
    ]
    unet_keys = [k for k in all_keys if 'unet' in k.lower()]
    lora_a    = [k for k in all_keys if 'lora_A' in k or 'lora_down' in k]
    lora_b    = [k for k in all_keys if 'lora_B' in k or 'lora_up' in k]

    print()
    print('--- Compatibility with FLUX.1-dev ---')
    print(f'  Total keys        : {len(all_keys)}')
    print(f'  FLUX-pattern keys : {len(flux_keys)}')
    print(f'  UNet-pattern keys : {len(unet_keys)}')
    print(f'  LoRA A/down keys  : {len(lora_a)}')
    print(f'  LoRA B/up keys    : {len(lora_b)}')

    if len(flux_keys) > 0 and len(unet_keys) == 0:
        print()
        print('  => FLUX-format LoRA: compatible with FLUX.1-dev')
    elif len(unet_keys) > 0 and len(flux_keys) == 0:
        print()
        print('  => UNet-only keys: trained for SD/SDXL, NOT compatible with FLUX.1-dev')
    elif len(flux_keys) > 0 and len(unet_keys) > 0:
        print()
        print('  => Mixed keys: partial compatibility, expect diffusers warnings on load')
    else:
        print()
        print('  => Unrecognised key format: compatibility cannot be determined from key names alone')
else:
    print('LoRA not found — LoRA test cells will print a skip message and continue.')


In [ ]:
_lora_loaded = False

if lora_exists:
    try:
        import peft.import_utils as _peft_utils
        import peft.tuners.lora.torchao as _peft_torchao
        _orig_fn = _peft_utils.is_torchao_available
        def _safe_fn():
            try:
                return _orig_fn()
            except ImportError:
                return False
        _peft_utils.is_torchao_available = _safe_fn
        _peft_torchao.is_torchao_available = _safe_fn
        print('PEFT torchao patch applied')

        pipe.load_lora_weights(
            os.path.dirname(LORA_SATIREFIC_PATH),
            weight_name=os.path.basename(LORA_SATIREFIC_PATH),
            adapter_name=LORA_SATIREFIC_ADAPTER_NAME,
        )
        pipe.disable_lora()
        _lora_loaded = True
        print(f'Satirefic LoRA loaded as adapter "{LORA_SATIREFIC_ADAPTER_NAME}"')
        print('LoRA disabled by default — activated per test below')
    except Exception as exc:
        print(f'ERROR loading Satirefic LoRA: {exc}')
        print('Continuing without LoRA.')
else:
    print('Satirefic LoRA not found — skipping load.')


## Prompt

Replicates `build_final_prompt()` from Notebook 05 exactly.  
Prints the complete final strings so there are no hidden transformations.

In [ ]:
def build_final_prompt(story_prompt, use_satirefic=False):
    """Replicates build_final_prompt() from Notebook 05 cell-inference."""
    base = story_prompt.strip()
    if not base.startswith(GLOBAL_PROMPT_PREFIX):
        base = GLOBAL_PROMPT_PREFIX + ', ' + base
    if use_satirefic and SATIREFIC_STYLE_BOOST not in base:
        base = base + ', ' + SATIREFIC_STYLE_BOOST
    return base


# Prompt A — used for diversity, repeatability, and LoRA tests
STORY_A = (
    'a bloated pig-politician in a suit handing out empty promises '
    'while human workers and families queue in the street'
)

# Prompt B — no people, no politics, entirely different subject and composition
STORY_B = 'a peaceful mountain cabin at sunrise, no people, soft clouds, simple landscape'

PROMPT_A     = build_final_prompt(STORY_A, use_satirefic=False)
PROMPT_A_SAT = build_final_prompt(STORY_A, use_satirefic=True)
PROMPT_B     = build_final_prompt(STORY_B, use_satirefic=False)

print('=== Prompt A (no LoRA) ===')
print(PROMPT_A)
print()
print('=== Prompt A (Satirefic boost) ===')
print(PROMPT_A_SAT)
print()
print('=== Prompt B (prompt-sensitivity test) ===')
print(PROMPT_B)


## Generate — Helpers

`sha256_pixels` hashes raw pixel bytes from the in-memory PIL object **before** PNG encoding.  
`sha256_file` hashes the saved file **after** PNG encoding.  
Filenames include task label, seed, and a UUID so no two runs can overwrite each other.

In [ ]:
def sha256_pixels(img):
    """SHA-256 of raw RGB pixel bytes — what the model produced in memory."""
    h = hashlib.sha256()
    h.update(img.tobytes())
    return h.hexdigest()


def sha256_file(path):
    """SHA-256 of the saved PNG file on disk."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()


def run_imagine(prompt, seed, label, guidance=3.5, steps=28):
    """
    Single FluxPipeline call with an isolated generator.
    Returns (PIL image, saved path, pixel_hash, file_hash).

    pixel_hash: SHA-256 of img.tobytes() BEFORE save.
                If this matches across runs the model produced the same output (seed/generator issue).
    file_hash:  SHA-256 of the .png file AFTER save.
                If pixel_hash differs but file_hash matches, a filename collision overwrote a file.
    """
    gen = torch.Generator(device='cuda').manual_seed(seed)

    print(f'  seed             : {seed}')
    print(f'  generator device : {gen.device}')
    print(f'  guidance         : {guidance}  steps: {steps}')
    print(f'  prompt[:100]     : {prompt[:100]}...')

    with torch.inference_mode():
        out = pipe(
            prompt=prompt,
            width=1024,
            height=1024,
            num_inference_steps=steps,
            guidance_scale=guidance,
            generator=gen,
            max_sequence_length=512,
        )

    img      = out.images[0]                    # PIL image from this pipeline call
    px_hash  = sha256_pixels(img)               # hash pixels BEFORE encoding

    uid      = uuid.uuid4().hex[:8]
    filename = f'06_{label}_seed{seed}_{uid}.png'
    path     = os.path.join(OUTPUT_DIR, filename)
    img.save(path)
    f_hash   = sha256_file(path)                # hash file AFTER encoding

    print(f'  filename         : {filename}')
    print(f'  pixel SHA-256    : {px_hash}  <- model output')
    print(f'  file  SHA-256    : {f_hash}  <- on disk')
    return img, path, px_hash, f_hash


## Test 1 — Diversity (No LoRA)

Same prompt, seeds 42 / 12345 / 987654. All three **pixel hashes** must differ.
Duplicate pixel hashes print every pipeline variable and raise `AssertionError`.

In [ ]:
SEEDS = [42, 12345, 987654]
seed_results = {}

if _lora_loaded:
    pipe.disable_lora()
print('=== Test 1: Diversity — same prompt, seeds 42 / 12345 / 987654, no LoRA ===')
print('Expected: all pixel hashes differ.')
print()

for seed in SEEDS:
    print(f'--- Seed {seed} ---')
    img, path, px_h, f_h = run_imagine(PROMPT_A, seed, 'diversity')
    seed_results[seed] = {'img': img, 'path': path, 'pixel_hash': px_h, 'file_hash': f_h}
    print()

print('=== Diversity Test Results ===')
for seed, r in seed_results.items():
    print(f"  Seed {seed:8d} | pixel: {r['pixel_hash'][:24]}... | file: {r['file_hash'][:24]}...")

unique_px = len(set(r['pixel_hash'] for r in seed_results.values()))
if unique_px == len(SEEDS):
    print()
    print(f'PASS: all {len(SEEDS)} seeds produced distinct pixel hashes')
else:
    print()
    print(f'FAIL: only {unique_px} distinct pixel hashes from {len(SEEDS)} seeds')

    seen_px, seen_f = {}, {}
    for seed, r in seed_results.items():
        if r['pixel_hash'] in seen_px:
            print(f"  Seed {seed} pixel_hash == Seed {seen_px[r['pixel_hash']]}  => model produced same output")
        seen_px[r['pixel_hash']] = seed
        if r['file_hash'] in seen_f:
            print(f"  Seed {seed} file_hash  == Seed {seen_f[r['file_hash']]}  => same bytes on disk")
        seen_f[r['file_hash']] = seed

    print()
    print('=== DEBUG: every variable passed to pipe() ===')
    print(f'  type(pipe)           : {type(pipe).__name__}')
    print(f'  prompt               : {PROMPT_A}')
    print(f'  width / height       : 1024 x 1024')
    print(f'  num_inference_steps  : 28')
    print(f'  guidance_scale       : 3.5')
    print(f'  generator            : fresh torch.Generator("cuda").manual_seed(seed) each loop')
    print(f'  max_sequence_length  : 512')
    raise AssertionError('Diversity test FAILED — review DEBUG output above before continuing.')


## Test 2 — Repeatability

Same prompt + seed 42, two consecutive runs. **Pixel hashes must match** — the pipeline must be deterministic.

In [ ]:
print('=== Test 2: Repeatability — same prompt, seed 42, two consecutive runs ===')
print('Expected: pixel hashes MATCH.')
print()

repro_runs = []
for run_n in [1, 2]:
    print(f'--- Run {run_n} ---')
    img, path, px_h, f_h = run_imagine(PROMPT_A, 42, f'repro_run{run_n}')
    repro_runs.append({'run': run_n, 'img': img, 'path': path, 'pixel_hash': px_h, 'file_hash': f_h})
    print()

print('=== Repeatability Result ===')
print(f"  Run 1 pixel: {repro_runs[0]['pixel_hash'][:32]}...")
print(f"  Run 2 pixel: {repro_runs[1]['pixel_hash'][:32]}...")
print(f"  Run 1 file : {repro_runs[0]['file_hash'][:32]}...")
print(f"  Run 2 file : {repro_runs[1]['file_hash'][:32]}...")
print()
if repro_runs[0]['pixel_hash'] == repro_runs[1]['pixel_hash']:
    print('PASS: same seed produces identical pixel output (pipeline is deterministic)')
else:
    print('FAIL: same seed produces different pixel output — pipeline is non-deterministic')
    print('  This means the generator alone does not fully control output.')
    print('  Check whether the scheduler uses a separate RNG source.')


## Test 3 — Prompt Sensitivity

Prompt B (mountain cabin at sunrise, no people) with seed 42. **Pixel hash must differ** from the seed-42 result in Test 1.

This confirms the model responds to the prompt content, not only to the seed.
The two subjects — a political crowd scene vs a solitary rural landscape — should produce visually distinct images.

In [ ]:
print('=== Test 3: Prompt Sensitivity — seed 42, Prompt A vs Prompt B ===')
print('Expected: pixel hashes DIFFER.')
print()

print('--- Prompt B (seed 42) ---')
img_b, path_b, px_h_b, f_h_b = run_imagine(PROMPT_B, 42, 'promptsensitivity')
print()

px_h_a = seed_results[42]['pixel_hash']
print('=== Prompt Sensitivity Result ===')
print(f'  Prompt A pixel: {px_h_a[:32]}...')
print(f'  Prompt B pixel: {px_h_b[:32]}...')
print()
if px_h_a != px_h_b:
    print('PASS: different prompt with same seed produces a different image')
else:
    print('FAIL: different prompt with same seed produces identical pixel output — prompt is being ignored')


## Generate — With Satirefic LoRA

Same three seeds, Prompt A with Satirefic boost applied. Skipped if the LoRA did not load.

In [ ]:
lora_results = {}

if _lora_loaded:
    pipe.set_adapters([LORA_SATIREFIC_ADAPTER_NAME], adapter_weights=[0.9])
    pipe.enable_lora()
    print('Satirefic LoRA activated at strength 0.9')
    print()

    for seed in SEEDS:
        print(f'--- Seed {seed} ---')
        img, path, px_h, f_h = run_imagine(PROMPT_A_SAT, seed, 'satirefic_diversity')
        lora_results[seed] = {'img': img, 'path': path, 'pixel_hash': px_h, 'file_hash': f_h}
        print()

    print('=== Satirefic Diversity Results ===')
    for seed, r in lora_results.items():
        print(f"  Seed {seed:8d} | pixel: {r['pixel_hash'][:24]}... | file: {r['file_hash'][:24]}...")

    unique_lora = len(set(r['pixel_hash'] for r in lora_results.values()))
    if unique_lora == len(SEEDS):
        print()
        print(f'PASS: all {len(SEEDS)} seeds produced distinct pixel hashes with Satirefic LoRA')
    else:
        print()
        print(f'FAIL: only {unique_lora} unique pixel hashes with LoRA — LoRA broke seed behaviour')

    pipe.disable_lora()
else:
    print('Satirefic LoRA not loaded — skipping LoRA test.')
    print(f'Place the file at: {LORA_SATIREFIC_PATH}')


## Verify Outputs

Display all generated images. Confirm visually that different seeds produce different compositions.

In [ ]:
from IPython.display import display as ipy_display


def tile_images(images, thumb=512):
    n = len(images)
    canvas = Image.new('RGB', (thumb * n, thumb), (20, 20, 24))
    for i, img in enumerate(images):
        canvas.paste(img.resize((thumb, thumb), Image.LANCZOS), (i * thumb, 0))
    return canvas


print('=== Test 1 — Diversity: seeds 42 / 12345 / 987654 (no LoRA) ===')
ipy_display(tile_images([seed_results[s]['img'] for s in SEEDS]))

print()
print('=== Test 2 — Repeatability: same prompt + seed 42 ===')
print('  Left : Run 1 — same prompt + seed 42')
print('  Right: Run 2 — same prompt + seed 42')
ipy_display(tile_images([repro_runs[0]['img'], repro_runs[1]['img']]))

print()
print('=== Test 3 — Prompt Sensitivity: seed 42 ===')
print('  Left : Prompt A — pig-politician crowd scene')
print('  Right: Prompt B — mountain cabin at sunrise, no people')
ipy_display(tile_images([seed_results[42]['img'], img_b]))

if lora_results:
    print()
    print('=== Satirefic LoRA: seeds 42 / 12345 / 987654 ===')
    ipy_display(tile_images([lora_results[s]['img'] for s in SEEDS]))


In [ ]:
seeds_distinct = len(set(r['pixel_hash'] for r in seed_results.values())) == len(SEEDS)
repro_ok       = repro_runs[0]['pixel_hash'] == repro_runs[1]['pixel_hash']
prompt_diff    = seed_results[42]['pixel_hash'] != px_h_b

print('=== Final Summary ===')
print()
print(f'Pipeline                                         : FluxPipeline')
print(f'Base model                                       : {BASE_MODEL}')
print(f'Satirefic LoRA file found                        : {lora_exists}')
print(f'Satirefic LoRA loaded                            : {_lora_loaded}')
print()
print(f'[DIVERSITY]      Different seeds -> distinct pixel hashes : {seeds_distinct}')
print(f'[REPEATABILITY]  Same seed -> same pixel hash (twice)     : {repro_ok}')
print(f'[PROMPT SENS.]   Different prompt -> different pixel hash  : {prompt_diff}')

if lora_results:
    lora_distinct = len(set(r['pixel_hash'] for r in lora_results.values())) == len(SEEDS)
    print(f'[LORA DIVERSITY] Satirefic: seeds produce distinct hashes  : {lora_distinct}')

print()
print('Pixel hashes (model output):')
for seed, r in seed_results.items():
    print(f"  diversity seed {seed:8d}: {r['pixel_hash'][:32]}...")
print(f"  repro run1 seed  42: {repro_runs[0]['pixel_hash'][:32]}...")
print(f"  repro run2 seed  42: {repro_runs[1]['pixel_hash'][:32]}...")
print(f"  prompt-B seed    42: {px_h_b[:32]}...")
if lora_results:
    for seed, r in lora_results.items():
        print(f"  satirefic seed {seed:8d}: {r['pixel_hash'][:32]}...")

print()
print('=== Compatibility Notes ===')
print('  Satirefic compatibility with FLUX.1-dev is unproven until runtime loading')
print('  and generation succeeds. A successful load does not by itself prove a visible')
print('  style effect — the LoRA test images must differ visually from the no-LoRA images.')
